# 01 - 探索的データ分析 (EDA)
**近赤外研究会 スペクトル分析チャレンジ**  
近赤外（NIR）スペクトルから木材の含水率を予測するコンペティション

## このノートブックの目的
- データの基本構造を把握する
- 含水率（ターゲット）の分布を確認する
- NIRスペクトルの形状と樹種ごとの違いを可視化する
- **Train/Testで樹種が完全に異なる**という重要な事実を確認する
- 水の吸収ピーク帯域を確認し、モデル設計の方針を立てる

## 0. ライブラリのインポート

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import japanize_matplotlib  # 日本語フォント対応

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('ライブラリ読み込み完了')

## 1. データの読み込み

In [ ]:
train = pd.read_csv('../data/train_near.csv', encoding='cp932')
test  = pd.read_csv('../data/test_near.csv',  encoding='cp932')

# スペクトル列（波数の数値列）とメタ列を分離
META_COLS  = ['sample number', 'species number', '樹種', '含水率']
SPEC_COLS  = [c for c in train.columns if c not in META_COLS]
WAVENUMS   = np.array([float(c) for c in SPEC_COLS])  # 波数 [cm⁻¹]
WAVELENGTHS = 10_000 / WAVENUMS * 1000                # 波長 [nm] に換算

print(f'Train : {train.shape}  ({len(SPEC_COLS)} スペクトル変数)')
print(f'Test  : {test.shape}')
print(f'波数範囲 : {WAVENUMS.max():.0f} – {WAVENUMS.min():.0f} cm⁻¹')
print(f'波長範囲 : {WAVELENGTHS.min():.0f} – {WAVELENGTHS.max():.0f} nm')

> **補足**: NIR（近赤外）領域は一般に 780〜2500 nm（= 12820〜4000 cm⁻¹）。  
> このデータは **1001〜2500 nm** をほぼカバーしており、木材水分分析に最適な帯域です。

## 2. 樹種の確認：Train と Test は完全に別種！

In [ ]:
train_species = set(train['樹種'].unique())
test_species  = set(test['樹種'].unique())

print('Train 樹種:', sorted(train_species))
print('Test  樹種:', sorted(test_species))
print()
print('重複:', train_species & test_species)  # 空集合になるはず
print('→ Train と Test の樹種は 1種類も重複していない！')

> ⚠️ **これはこのコンペ最大の特徴です。**  
> 樹種を「丸暗記」するモデルは Test で機能しません。  
> **NIRスペクトルの物理的シグナル（特に水のO-H吸収帯）**から含水率を汎化的に学習する必要があります。  
> → 樹種ダミー変数はモデルに使わない方針とします。

## 3. ターゲット（含水率）の分布

In [ ]:
y = train['含水率']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 生の分布
axes[0].hist(y, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(y.mean(),   color='red',    linestyle='--', label=f'平均 {y.mean():.1f}%')
axes[0].axvline(y.median(), color='orange', linestyle='--', label=f'中央値 {y.median():.1f}%')
axes[0].set_xlabel('含水率 [%]')
axes[0].set_ylabel('頻度')
axes[0].set_title('含水率の分布（生値）')
axes[0].legend()

# 対数変換後
axes[1].hist(np.log1p(y), bins=50, color='coral', edgecolor='white')
axes[1].set_xlabel('log(1 + 含水率)')
axes[1].set_ylabel('頻度')
axes[1].set_title('含水率の分布（log1p 変換後）')

plt.tight_layout()
plt.savefig('figs/01_target_distribution.png', bbox_inches='tight')
plt.show()

print(y.describe())

> **観察**: 含水率は右裾が長い分布（0.84〜298%）。  
> log変換するとほぼ正規分布に近づきます。  
> → モデリング時に **log1p 変換したターゲット**で学習し、予測後に `expm1` で戻す戦略が有効かもしれません。

## 4. 樹種別の含水率分布

In [ ]:
# 樹種を中央値順にソート
species_order = train.groupby('樹種')['含水率'].median().sort_values().index

fig, ax = plt.subplots(figsize=(12, 5))
data_by_species = [train[train['樹種'] == s]['含水率'].values for s in species_order]

bp = ax.boxplot(data_by_species, labels=species_order, patch_artist=True, notch=False)
colors = cm.tab20(np.linspace(0, 1, len(species_order)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel('樹種')
ax.set_ylabel('含水率 [%]')
ax.set_title('樹種別 含水率分布（Train）')
plt.tight_layout()
plt.savefig('figs/01_species_moisture.png', bbox_inches='tight')
plt.show()

## 5. NIRスペクトルの可視化

In [ ]:
# 各樹種から3サンプルずつ抽出してスペクトルを重ね描き
fig, ax = plt.subplots(figsize=(14, 5))

colors = cm.tab20(np.linspace(0, 1, len(train_species)))
plotted_species = set()

for i, species in enumerate(sorted(train_species)):
    subset = train[train['樹種'] == species].head(3)
    for j, (_, row) in enumerate(subset.iterrows()):
        label = species if j == 0 else None  # 凡例は1本だけ
        ax.plot(WAVELENGTHS, row[SPEC_COLS].values,
                color=colors[i], alpha=0.6, linewidth=0.8, label=label)

# 水の主要吸収帯をハイライト
water_bands = [
    (1350, 1550, '水 O-H 倍音\n(~1450 nm)'),
    (1850, 2050, '水 O-H 結合音\n(~1940 nm)'),
]
for wl_min, wl_max, label in water_bands:
    ax.axvspan(wl_min, wl_max, alpha=0.12, color='blue')
    ax.text((wl_min + wl_max)/2, ax.get_ylim()[1]*0.9, label,
            ha='center', fontsize=8, color='navy')

ax.set_xlabel('波長 [nm]')
ax.set_ylabel('吸光度')
ax.set_title('NIRスペクトル（Train・樹種別3サンプル）')
ax.legend(loc='upper right', fontsize=7, ncol=3)
ax.invert_xaxis()  # 慣例として波数増加方向 → 波長短波長方向に
plt.tight_layout()
plt.savefig('figs/01_spectra_overview.png', bbox_inches='tight')
plt.show()

> **ポイント**: 青いハイライト帯域（1450 nm・1940 nm 付近）が水分子の O-H 結合の吸収帯です。  
> 含水率が高いサンプルはここで吸光度が大きくなる傾向があります。

## 6. 含水率とスペクトルの関係

In [ ]:
# 含水率でサンプルを5グループに分けてスペクトル平均を比較
train['mc_group'] = pd.qcut(train['含水率'], q=5,
                             labels=['極低(<10%)', '低', '中', '高', '極高(>100%)'])

fig, ax = plt.subplots(figsize=(14, 5))
colors_mc = ['#2166ac', '#74add1', '#fee090', '#f46d43', '#d73027']

for grp, color in zip(['極低(<10%)', '低', '中', '高', '極高(>100%)'], colors_mc):
    subset = train[train['mc_group'] == grp]
    mean_spec = subset[SPEC_COLS].mean().values
    ax.plot(WAVELENGTHS, mean_spec, color=color, linewidth=1.5,
            label=f'{grp} (n={len(subset)})')

# 水の吸収帯
for wl_min, wl_max, label in water_bands:
    ax.axvspan(wl_min, wl_max, alpha=0.12, color='blue')

ax.set_xlabel('波長 [nm]')
ax.set_ylabel('平均吸光度')
ax.set_title('含水率グループ別 平均NIRスペクトル')
ax.legend(fontsize=9)
ax.invert_xaxis()
plt.tight_layout()
plt.savefig('figs/01_spectra_by_mc_group.png', bbox_inches='tight')
plt.show()

> **観察**: 含水率が高いグループほど、水の吸収帯（1450・1940 nm）での吸光度が大きくなっていれば、  
> NIRスペクトルから含水率を予測できる根拠になります。

## 7. 各波長での含水率との相関係数

In [ ]:
# 各波長の吸光度と含水率のピアソン相関を計算
correlations = train[SPEC_COLS].corrwith(train['含水率'])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(WAVELENGTHS, correlations.values, linewidth=0.8, color='darkgreen')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

# 水の吸収帯
for wl_min, wl_max, label in water_bands:
    ax.axvspan(wl_min, wl_max, alpha=0.15, color='blue')
    ax.text((wl_min + wl_max)/2, 0.05, label.split('\n')[0],
            ha='center', fontsize=8, color='navy')

ax.set_xlabel('波長 [nm]')
ax.set_ylabel('ピアソン相関係数')
ax.set_title('各波長での吸光度と含水率の相関')
ax.invert_xaxis()
plt.tight_layout()
plt.savefig('figs/01_correlation_per_wavelength.png', bbox_inches='tight')
plt.show()

top5 = correlations.abs().nlargest(5)
print('相関が高い上位5波長:')
for wn, corr in zip(top5.index, correlations[top5.index]):
    wl = 10000 / float(wn) * 1000
    print(f'  {wn} cm⁻¹ ({wl:.0f} nm) : r = {corr:.4f}')

## 8. まとめ：EDAから得られた知見

| 項目 | 内容 |
|------|------|
| **Train/Test の樹種** | 完全に別種 → 樹種をモデルの特徴量に使ってはいけない |
| **含水率の分布** | 右裾が長い（0.84〜298%）→ log1p 変換を検討 |
| **スペクトル変数数** | 1555 変数（波数 4000〜9994 cm⁻¹、波長 1001〜2500 nm） |
| **水の吸収帯** | 1450 nm・1940 nm 付近が含水率と高い相関を持つと予想 |
| **モデル方針** | スペクトルの物理的シグナルを活かす前処理（SNV・微分）が重要 |

### 次のステップ
- `02_preprocessing.ipynb`: SNV・Savitzky-Golay微分などの前処理を実装・比較
- `03_baseline_pls.ipynb`: PLS回帰でベースラインスコアを確立